In [1]:
import os
print(os.getcwd())

/home/sagemaker-user


In [2]:
import json
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch


In [3]:
# ✅ Base chat model
TUNE_MODEL = "mistral7b_subject_merged"
DATA_PATH = "DPO_DATA_VER1.jsonl"

In [4]:
# ---- Load dataset ----
prompts, completions = [], []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        line = line.strip()

        # Skip empty line
        if not line:
            continue

        # Remove BOM
        if line.startswith("\ufeff"):
            line = line.replace("\ufeff", "")

        try:
            d = json.loads(line)
        except json.JSONDecodeError as e:
            print(f"❌ JSON error in line {i}: {e}")
            print("Line content:", repr(line))
            continue 

        prompts.append(d["prompt"])
        completions.append(d["completion"])

print(f"Loaded {len(prompts)} samples")

Loaded 520 samples


In [5]:
# ---- Load model ----
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # A10G supports bf16
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    TUNE_MODEL,
    use_fast=True,
    local_files_only=True
)

# Leave headroom so auto-sharding never tries CPU/disk
max_mem = {0: "22GiB", "cpu": "0GiB"}  # restrict CPU offload explicitly

model = AutoModelForCausalLM.from_pretrained(
    TUNE_MODEL,
    quantization_config=bnb_cfg,
    device_map="auto",
    max_memory=max_mem,
    low_cpu_mem_usage=True,
    local_files_only=True,
)

model.eval()
print("Loaded on:", next(model.parameters()).device)

2025-12-03 20:01:33.805963: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764792093.829567   13006 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764792093.837350   13006 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-03 20:01:33.870587: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded on: cuda:0


In [6]:
# ---- Chat prompt format for BioMistral ----
def build_prompt(symptoms):
    return f"""
### Instruction:
You are a medical diagnosis assistant. Based on the given patient symptoms, output the most likely disease and the appropriate medical department.

Return answer STRICTLY in this format:

Disease: <disease name>
Department: <department name>

If unsure, output the most reasonable guess based on common medical knowledge.

Symptoms: {symptoms}

### Response:
""".strip()


In [7]:
import re

def format_output(text):
    # normalize lowercase
    text = text.strip()

    disease = "Unknown"
    department = "Unknown"

    # extract disease
    m = re.search(r"Disease\s*:\s*(.+)", text, re.IGNORECASE)
    if m:
        disease = m.group(1).strip()

    # extract department
    m = re.search(r"Department\s*:\s*(.+)", text, re.IGNORECASE)
    if m:
        department = m.group(1).strip()

    return f"Disease: {disease}\nDepartment: {department}"

In [8]:
def generate_tune(symptoms):
    prompt = build_prompt(symptoms)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    text = tokenizer.decode(output[0], skip_special_tokens=True)

    if text.startswith(prompt):
        text = text[len(prompt):]

    return format_output(text)

In [10]:
# ---- Run fine-tuned model on first N samples ----
N = 100
tune_outputs = [generate_tune(p) for p in prompts[:N]]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

In [14]:
# ---- Save result ----
df = pd.DataFrame({
    "symptom_prompt": prompts[:N],
    "tune_output": tune_outputs,
    "expected_completion": completions[:N]
})

df.to_csv("before-DPO-results_biomistral.csv", index=False)
df.head()

,symptom_prompt,tune_output,expected_completion
0,### Instruction:\nGiven the following symptoms...,Disease: urinary tract infection\nDepartment: ...,Disease: urinary tract infection\nDepartment: ...
1,### Instruction:\nGiven the following symptoms...,Disease: allergy\nDepartment: dermatology / al...,Disease: allergy\nDepartment: Dermatology / Al...
2,### Instruction:\nGiven the following symptoms...,Disease: pneumonia\nDepartment: Pulmonology / ...,Disease: pneumonia\nDepartment: Pulmonology / ...
3,### Instruction:\nGiven the following symptoms...,Disease: common cold\nDepartment: Internal Med...,Disease: common cold\nDepartment: Internal Med...
4,### Instruction:\nGiven the following symptoms...,Disease: common cold\nDepartment: Internal Med...,Disease: common cold\nDepartment: Internal Med...


In [19]:
import pandas as pd
import nltk
import re
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sklearn.metrics import f1_score

# Download NLTK tokenizer
nltk.download('punkt')
nltk.download('punkt_tab')

# Load CSV
df = pd.read_csv("before-DPO-results_biomistral.csv")

smooth = SmoothingFunction().method3
bleu_scores = []
accuracy_scores = []

# ---------- Helper: extract Disease / Department ----------
def extract_label(text, key):
    """
    Extract the substring after "Disease:" or "Department:" (case insensitive).
    """
    pattern = rf"{key}\s*:\s*(.+)"
    match = re.search(pattern, str(text), re.IGNORECASE)
    if match:
        # normalize: remove extra spaces + lowercase
        return match.group(1).strip().lower()
    return ""

# ---------- Compute BLEU + Exact Match ----------
for ref, pred in zip(df["expected_completion"], df["tune_output"]):

    # Tokenize for BLEU
    ref_tokens = nltk.word_tokenize(str(ref))
    pred_tokens = nltk.word_tokenize(str(pred))

    score = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smooth)
    bleu_scores.append(score)

    # Exact match accuracy
    accuracy = 1 if str(ref).strip() == str(pred).strip() else 0
    accuracy_scores.append(accuracy)

df["BLEU"] = bleu_scores
df["ExactMatch"] = accuracy_scores

# ---------- Extract labels for F1 ----------
df["true_disease"] = df["expected_completion"].apply(lambda x: extract_label(x, "Disease"))
df["true_department"] = df["expected_completion"].apply(lambda x: extract_label(x, "Department"))

df["pred_disease"] = df["tune_output"].apply(lambda x: extract_label(x, "Disease"))
df["pred_department"] = df["tune_output"].apply(lambda x: extract_label(x, "Department"))

# ---------- Compute F1 ----------
disease_f1 = f1_score(df["true_disease"], df["pred_disease"], average="macro")
department_f1 = f1_score(df["true_department"], df["pred_department"], average="macro")

# ---------- Print results ----------
print(f"Average BLEU: {df['BLEU'].mean():.4f}")
print(f"Exact Match Accuracy: {df['ExactMatch'].mean():.4f}")
print(f"Disease Macro F1: {disease_f1:.4f}")
print(f"Department Macro F1: {department_f1:.4f}")

Average BLEU: 0.7069
Exact Match Accuracy: 0.2500
Disease Macro F1: 0.9577
Department Macro F1: 0.0786


[nltk_data] Downloading package punkt to /home/sagemaker-
[nltk_data]     user/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/sagemaker-
[nltk_data]     user/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
